# Queens 8 × 8 · Discrete Flow Matching

One model learns to turn a complete coloring into **queen colors + EMPTY**.
The complete coloring is the initial state. The model predicts endpoint erasure from the current grid.

Train on our prepared synthetic puzzles, inspect solution trajectories, and measure
the probability of finding at least one valid solution within a given attempt budget.

In [6]:
from pathlib import Path
import sys
import hashlib

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from _shared.queens_data import BOARD_SIZE, EMPTY, load_queens_dataset, solution_checks
from _shared.queens_dfm_training import QueensDFMRun
from _shared.queens_dfm_inference import sample_queens_dfm

SEED = 17
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FLOW_STEPS = 100
ARTIFACTS = PROJECT_ROOT / "artifacts" / "queens8"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACTS / "queens8_dfm.pt"
NUM_CLASSES = BOARD_SIZE + 1
PALETTE = [
    "#B8A5E8",
    "#F0B48B",
    "#92C9E8",
    "#A9D4A4",
    "#E8CD83",
    "#ECA6BB",
    "#9FB9D4",
    "#C7B8A3",
    "#F1F3F5",
]
CATEGORICAL_SCALE = [
    [position, color]
    for index, color in enumerate(PALETTE)
    for position in (index / NUM_CLASSES, (index + 1) / NUM_CLASSES)
]
PLOT_CONFIG = dict(responsive=True, displaylogo=False, displayModeBar=False)
display(HTML(f"<b>Queens 8 × 8 · DFM</b> · {DEVICE} · {FLOW_STEPS} sampling steps"))

## 1. Synthetic dataset

Puzzle/solution pairs are prepared offline. All eight regions are connected,
contain 3–14 cells, and each puzzle has **exactly one solution**, verified exhaustively.

Place one queen in each row, column and region. Queens must not touch diagonally;
longer diagonals are allowed. Split puzzles before augmentation; rotations, reflections
and color permutations cannot leak duplicate puzzles between splits.

The notebook only reads `data/queens8/`. To prepare another dataset separately, run
`uv run python scripts/generate_queens8.py --count 5000`.

In [7]:
puzzles = load_queens_dataset(PROJECT_ROOT / "data" / "queens8")
regions = puzzles["regions"]
display(
    HTML(
        f"<b>{len(regions):,} unique puzzles</b> · synthetic · {BOARD_SIZE} × {BOARD_SIZE}"
    )
)

In [8]:
def add_board(fig, tokens, board, row, col, queens=False):
    tokens, board = np.asarray(tokens), np.asarray(board)
    fig.add_trace(
        go.Heatmap(
            z=tokens,
            zmin=-0.5,
            zmax=NUM_CLASSES - 0.5,
            colorscale=CATEGORICAL_SCALE,
            showscale=False,
            xgap=1,
            ygap=1,
            customdata=board,
            hovertemplate="row=%{y}, col=%{x}<br>region=%{customdata}<br>token=%{z}<extra></extra>",
        ),
        row=row,
        col=col,
    )
    xs, ys = [], []
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if c + 1 < BOARD_SIZE and board[r, c] != board[r, c + 1]:
                xs.extend([c + 0.5, c + 0.5, None])
                ys.extend([r - 0.5, r + 0.5, None])
            if r + 1 < BOARD_SIZE and board[r, c] != board[r + 1, c]:
                xs.extend([c - 0.5, c + 0.5, None])
                ys.extend([r + 0.5, r + 0.5, None])
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color="#64748B", width=1.5),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if queens:
        qr, qc = np.where(tokens != EMPTY)
        fig.add_trace(
            go.Scatter(
                x=qc,
                y=qr,
                mode="text",
                text=["♛"] * len(qr),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
    fig.update_xaxes(
        range=[-0.5, BOARD_SIZE - 0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )
    xaxis_name = fig.get_subplot(row, col).xaxis.plotly_name.replace("axis", "")
    fig.update_yaxes(
        range=[BOARD_SIZE - 0.5, -0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        scaleanchor=xaxis_name,
        scaleratio=1,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )


def style_boards(fig, title, rows=1):
    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=22)),
        template="plotly_white",
        autosize=True,
        height=rows * 225 + 90,
        margin=dict(l=10, r=10, t=85, b=20),
        font=dict(family="Arial, sans-serif", size=15),
        showlegend=False,
    )
    fig.update_annotations(font=dict(size=14))
    return fig

In [9]:
N_DATASET_EXAMPLES = 6
example_indices = np.random.default_rng(SEED + 101).choice(
    len(regions),
    size=min(N_DATASET_EXAMPLES, len(regions)),
    replace=False,
)
example_cols = min(3, len(example_indices))
example_rows = (len(example_indices) + example_cols - 1) // example_cols
examples_fig = make_subplots(
    rows=example_rows,
    cols=example_cols,
    subplot_titles=[f"Example {number + 1}" for number in range(len(example_indices))],
    horizontal_spacing=0.055,
    vertical_spacing=0.10,
)
for number, index in enumerate(example_indices):
    row, col = divmod(number, example_cols)
    board = regions[index]
    add_board(examples_fig, board, board, row + 1, col + 1)
    queen_rows, queen_cols = np.where(puzzles["queens"][index])
    examples_fig.add_trace(
        go.Scatter(
            x=queen_cols,
            y=queen_rows,
            mode="text",
            text=["♛"] * len(queen_rows),
            textfont=dict(size=28, color="#172033"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row + 1,
        col=col + 1,
    )
style_boards(examples_fig, "Dataset examples · stored solutions", rows=example_rows)
examples_fig.update_layout(height=example_rows * 290 + 90)
examples_fig.show(config=PLOT_CONFIG)

## 2. Train DFM

The full coloring is $x_0=c$. The endpoint $x_1$ keeps $c$ only at queen positions
and uses EMPTY elsewhere. For each augmented training pair, sample $t\sim U[0,1)$
and independent per-cell switches:

$$
x_t^i =
\begin{cases}
x_1^i, & u_i<t,\\
c^i, & u_i\geq t,
\end{cases}
\qquad u_i\sim U[0,1).
$$

The U-Net receives **nine one-hot state channels** and two coordinate channels.
`USE_TIME_CONDITIONING` controls its optional time branch. When enabled,
sinusoidal features pass through an MLP and learned projections in each
residual block. When disabled, these modules are absent and prediction
takes only the grid.

The output is **one erasure logit per cell**, shaped $[B,1,8,8]$.
The endpoint EMPTY probability is $p_i=\sigma(\ell_i)$.

The binary target is $y_i=\mathbf{1}[x_1^i=\mathrm{EMPTY}]$: 1 for an empty
endpoint cell and 0 for a queen. Let $a_i=\mathbf{1}[x_t^i\ne\mathrm{EMPTY}]$.
Train with binary cross-entropy directly from logits:

$$
\mathcal L=\mathbb E\!\left[\frac{1}{64}\sum_{i=1}^{64}
a_i\left(\operatorname{softplus}(\ell_i)-y_i\ell_i\right)\right].
$$

`binary_cross_entropy_with_logits` evaluates the loss stably. Already-empty cells
contribute zero; the mean includes all cells and batch samples. During sampling,
erased cells remain EMPTY and retained cells keep their current region colors.

One live plot updates during training. The checkpoint stores weights, optimizer,
history, splits and RNG state; setup automatically resumes it. `EPOCHS` is the
**total budget**: increase it to continue training. Saving happens every 20 epochs
and when training ends or is interrupted.
The time-conditioning flag is part of the saved architecture. Set `RESUME = False`
to train a different architecture at the checkpoint path; use `RESUME = True`
to continue a matching model. Generation-step counts are configured in the
sampling cells.


In [10]:
EPOCHS = 5_000
BATCH_SIZE = 128
SAMPLES_PER_EPOCH = 4096
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
USE_TIME_CONDITIONING = False
RESUME = True

run = QueensDFMRun(
    puzzles,
    CHECKPOINT_PATH,
    device=DEVICE,
    seed=SEED,
    batch_size=BATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_EPOCH,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    use_time_conditioning=USE_TIME_CONDITIONING,
    resume=RESUME,
)
model, splits = run.model, run.splits
display(
    HTML(
        f"<b>{sum(p.numel() for p in model.parameters()):,} parameters</b> · "
        f"{'resumed' if run.resumed else 'new model'} · {run.global_step:,}/{run.target_steps:,} updates<br>"
        + f"Time conditioning: {model.use_time_conditioning}<br>"
        + " · ".join(f"{name}: {len(indices):,}" for name, indices in splits.items())
    )
)

In [11]:
TARGET_STEPS = int(EPOCHS) * run.steps_per_epoch
training_progress = widgets.IntProgress(
    value=run.global_step,
    max=max(TARGET_STEPS, run.global_step, 1),
    description="Training",
    layout=widgets.Layout(width="100%"),
)
training_status = widgets.HTML()
loss_plot = go.FigureWidget()
loss_plot.add_scatter(
    name="Train BCE · moving mean", mode="lines", line=dict(color="#5267DB", width=2.5)
)
loss_plot.add_scatter(
    name="Validation BCE", mode="lines", line=dict(color="#D88437", width=2.5)
)
loss_plot.update_layout(
    template="plotly_white",
    height=450,
    autosize=True,
    xaxis_title="Optimizer step",
    yaxis_title="Mean grid binary cross-entropy",
    yaxis_type="log",
    font=dict(family="Arial, sans-serif", size=17),
    legend=dict(orientation="h", x=0, y=1.12),
    margin=dict(l=75, r=25, t=55, b=65),
)
display(training_progress, training_status, loss_plot)


def update_training(run, event, target):
    if event == "step" and run.global_step % 25 and run.global_step != 1:
        return
    training_progress.value = run.global_step
    history = run.history
    values = np.asarray(history["train_bce"], dtype=float)
    window = min(25, len(values))
    with loss_plot.batch_update():
        if window:
            loss_plot.data[0].x = history["step"][window - 1 :]
            loss_plot.data[0].y = np.maximum(
                np.convolve(values, np.ones(window) / window, mode="valid"),
                1e-12,
            )
        loss_plot.data[1].x = history["validation_step"]
        loss_plot.data[1].y = np.maximum(history["validation_bce"], 1e-12)
    training_status.value = (
        f"Step {run.global_step:,} / {target:,} · "
        f"{history['epochs_completed']} completed epochs"
    )
    if event == "complete":
        training_progress.bar_style = "success"
        training_status.value += " · checkpoint saved / up to date"


update_training(run, "setup", TARGET_STEPS)
try:
    run.train(target_steps=TARGET_STEPS, progress=update_training)
except KeyboardInterrupt:
    pass

IntProgress(value=0, description='Training', layout=Layout(width='100%'), max=160000)

HTML(value='')

FigureWidget({
    'data': [{'line': {'color': '#5267DB', 'width': 2.5},
              'mode': 'lines',
              'name': 'Train BCE · moving mean',
              'type': 'scatter',
              'uid': '2bd1211b-da30-4f24-a403-14469a6c3f38'},
             {'line': {'color': '#D88437', 'width': 2.5},
              'mode': 'lines',
              'name': 'Validation BCE',
              'type': 'scatter',
              'uid': '4dffa02a-fa06-4611-b361-77ef4f3d772e'}],
    'layout': {'autosize': True,
               'font': {'family': 'Arial, sans-serif', 'size': 17},
               'height': 450,
               'legend': {'orientation': 'h', 'x': 0, 'y': 1.12},
               'margin': {'b': 65, 'l': 75, 'r': 25, 't': 55},
               'template': '...',
               'xaxis': {'title': {'text': 'Optimizer step'}},
               'yaxis': {'title': {'text': 'Mean grid binary cross-entropy'}, 'type': 'log'}}
})

## 3. Solution trajectories

Start from the complete coloring. At each step, evaluate $p_\theta(x_1\mid x_t,t)$
and independently erase each still-colored cell with probability

$$
\frac{\Delta t}{1-t}\,p_\theta(x_1^i=\mathrm{EMPTY}\mid x_t,t).
$$

Each panel shows one sampled trajectory on a held-out puzzle. Region outlines
identify the puzzle regions throughout the trajectory. The model receives
the remaining colors and EMPTY cells at each step.

In [12]:
if run.global_step < 1:
    raise RuntimeError("Train or load the model before sampling.")
N_TRAJECTORY_PUZZLES = 3
TRAJECTORY_ATTEMPTS = 2
CAPTURE_STEPS = tuple(sorted(set(np.rint(np.linspace(0, FLOW_STEPS, 6)).astype(int))))
for number, index in enumerate(splits["test"][:N_TRAJECTORY_PUZZLES]):
    board = regions[int(index)]
    initial_states = np.repeat(board[None], TRAJECTORY_ATTEMPTS, axis=0)
    prediction, snapshots = sample_queens_dfm(
        model,
        initial_states,
        steps=FLOW_STEPS,
        seed=SEED + 600 + number,
        capture_steps=CAPTURE_STEPS,
    )
    solved = solution_checks(initial_states, prediction.numpy())["solved"]
    titles = [
        (
            f"t = {step / FLOW_STEPS:.1f}"
            if step < FLOW_STEPS
            else ("Solved" if solved[attempt] else "Not solved")
        )
        for attempt in range(TRAJECTORY_ATTEMPTS)
        for step in CAPTURE_STEPS
    ]
    fig = make_subplots(
        rows=TRAJECTORY_ATTEMPTS,
        cols=len(CAPTURE_STEPS),
        subplot_titles=titles,
        horizontal_spacing=0.025,
        vertical_spacing=0.12,
    )
    for attempt in range(TRAJECTORY_ATTEMPTS):
        for col, step in enumerate(CAPTURE_STEPS, 1):
            add_board(
                fig,
                snapshots[step][attempt].numpy(),
                board,
                attempt + 1,
                col,
                queens=step == FLOW_STEPS,
            )
    style_boards(
        fig, f"DFM · {puzzles['ids'][int(index)]}", rows=TRAJECTORY_ATTEMPTS
    ).show(
        config=PLOT_CONFIG,
    )

## 4. Probability of at least one solution

Evaluate **1, 2, 4, 8, 16, 32 and 64 generation steps** separately. For every step
budget, use the same held-out puzzles and run **32 stochastic attempts per puzzle**.
Each attempt starts from the complete coloring and follows the selected time grid.
Check every result against the Queens rules.

If $c$ out of $A=32$ attempts succeeded, average over all subsets of $k$ recorded
attempts:

$$
\widehat{\mathrm{pass@}k}=1-\frac{\binom{A-c}{k}}{\binom{A}{k}},
\qquad 0\leq k\leq A.
$$

The curve averages this estimate over held-out puzzles. Its shaded 95% interval
bootstraps whole puzzles, keeping each puzzle's attempts together. At $k=32$, the
curve is the observed fraction of puzzles solved at least once in the 32 attempts.

Show all seven curves on one plot. Colors identify generation-step budgets;
matching shaded bands show the 95% intervals. Toggle a legend entry to
hide or show both its curve and band.


In [13]:
N_EVAL_PUZZLES = 256  # None evaluates every held-out puzzle.
ATTEMPTS_PER_PUZZLE = 32
EVAL_STEP_COUNTS = (1, 2, 4, 8, 16, 32, 64)
EVAL_BATCH_SIZE = 128
EVAL_SEED = SEED + 10_000
if run.global_step < 1:
    raise RuntimeError("Train or load the model before evaluation.")
if ATTEMPTS_PER_PUZZLE < 1 or EVAL_BATCH_SIZE < 1:
    raise ValueError("Attempt count and batch size must be positive.")
if N_EVAL_PUZZLES is not None and N_EVAL_PUZZLES < 1:
    raise ValueError("N_EVAL_PUZZLES must be positive or None.")
if (
    not EVAL_STEP_COUNTS
    or len(set(EVAL_STEP_COUNTS)) != len(EVAL_STEP_COUNTS)
    or any(not isinstance(steps, int) or steps < 1 for steps in EVAL_STEP_COUNTS)
):
    raise ValueError("Generation-step budgets must be distinct positive integers.")
run.save()
checkpoint_sha256 = hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest()
evaluation_indices = np.random.default_rng(EVAL_SEED).permutation(splits["test"])
if N_EVAL_PUZZLES is not None:
    evaluation_indices = evaluation_indices[:N_EVAL_PUZZLES]
flat_indices = np.repeat(evaluation_indices, ATTEMPTS_PER_PUZZLE)

evaluation_progress = widgets.IntProgress(
    value=0,
    max=len(flat_indices) * sum(EVAL_STEP_COUNTS),
    description="Evaluation",
    layout=widgets.Layout(width="100%"),
)
evaluation_status = widgets.HTML()
display(evaluation_progress, evaluation_status)
evaluation_successes = {}
completed_work = 0
for budget_index, generation_steps in enumerate(EVAL_STEP_COUNTS, 1):
    success_batches = []
    for start in range(0, len(flat_indices), EVAL_BATCH_SIZE):
        indices = flat_indices[start : start + EVAL_BATCH_SIZE]
        evaluation_status.value = (
            f"{generation_steps} generation steps · budget {budget_index}/{len(EVAL_STEP_COUNTS)} · "
            f"{start:,} / {len(flat_indices):,} trajectories"
        )

        def update_evaluation(
            done,
            total,
            start=start,
            count=len(indices),
            offset=completed_work,
            steps=generation_steps,
        ):
            if done % 5 == 0 or done == total:
                evaluation_progress.value = offset + start * steps + done * count

        # Same puzzles and seed schedule across budgets; each is a fresh run
        # using its own time grid, not a truncation of a longer trajectory.
        prediction = sample_queens_dfm(
            model,
            regions[indices],
            steps=generation_steps,
            seed=EVAL_SEED + start,
            progress=update_evaluation,
        )
        success_batches.append(
            solution_checks(regions[indices], prediction.numpy())["solved"]
        )
    evaluation_successes[generation_steps] = np.concatenate(success_batches).reshape(
        len(evaluation_indices),
        ATTEMPTS_PER_PUZZLE,
    )
    completed_work += len(flat_indices) * generation_steps
    evaluation_progress.value = completed_work
evaluation_progress.bar_style = "success"
evaluation_status.value = (
    f"Completed {len(EVAL_STEP_COUNTS)} step budgets · {len(evaluation_indices):,} puzzles · "
    f"{ATTEMPTS_PER_PUZZLE} attempts each · "
    f"{len(flat_indices) * len(EVAL_STEP_COUNTS):,} total trajectories"
)

IntProgress(value=0, description='Evaluation', layout=Layout(width='100%'), max=1040384)

HTML(value='')

In [14]:
attempt_counts = np.arange(ATTEMPTS_PER_PUZZLE + 1)
bootstrap_rng = np.random.default_rng(EVAL_SEED + 1)
# Reuse the same whole-puzzle bootstrap draws for every generation-step budget.
n_puzzles = len(evaluation_indices)
weights = bootstrap_rng.multinomial(
    n_puzzles,
    np.full(n_puzzles, 1 / n_puzzles),
    size=3000,
)
retry_results = {}
step_colors = [
    "#6D28D9",
    "#4F46E5",
    "#2563EB",
    "#0891B2",
    "#059669",
    "#CA8A04",
    "#EA580C",
]
fig = go.Figure()
for generation_steps in EVAL_STEP_COUNTS:
    successes = evaluation_successes[generation_steps]
    per_puzzle_pass = np.zeros((n_puzzles, ATTEMPTS_PER_PUZZLE + 1))
    no_solution = np.ones(n_puzzles)
    failed = ATTEMPTS_PER_PUZZLE - successes.sum(axis=1)
    for k in range(1, ATTEMPTS_PER_PUZZLE + 1):
        no_solution *= np.maximum(failed - k + 1, 0) / (ATTEMPTS_PER_PUZZLE - k + 1)
        per_puzzle_pass[:, k] = 1 - no_solution
    pass_at_k = per_puzzle_pass.mean(axis=0)
    lower, upper = np.quantile(
        weights @ per_puzzle_pass / n_puzzles, [0.025, 0.975], axis=0
    )
    retry_results[generation_steps] = dict(
        pass_at_k=pass_at_k, lower=lower, upper=upper
    )

    color = step_colors[EVAL_STEP_COUNTS.index(generation_steps) % len(step_colors)]
    rgb = tuple(int(color[position : position + 2], 16) for position in (1, 3, 5))
    group = str(generation_steps)
    fig.add_scatter(
        x=attempt_counts,
        y=100 * lower,
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
        legendgroup=group,
    )
    fig.add_scatter(
        x=attempt_counts,
        y=100 * upper,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=f"rgba({rgb[0]},{rgb[1]},{rgb[2]},0.08)",
        hoverinfo="skip",
        showlegend=False,
        legendgroup=group,
    )
    fig.add_scatter(
        x=attempt_counts,
        y=100 * pass_at_k,
        mode="lines+markers",
        line=dict(color=color, width=2.5),
        marker=dict(size=5),
        name=f"{generation_steps} " + ("step" if generation_steps == 1 else "steps"),
        legendgroup=group,
        customdata=np.column_stack((100 * lower, 100 * upper)),
        hovertemplate=(
            "%{y:.2f}% · 95% interval [%{customdata[0]:.2f}%, %{customdata[1]:.2f}%]"
            "<extra>%{fullData.name}</extra>"
        ),
    )

fig.update_layout(
    title=dict(text="At least one solution vs number of attempts", x=0),
    template="plotly_white",
    height=580,
    autosize=True,
    hovermode="x unified",
    xaxis=dict(
        title="Number of attempts",
        range=[0, ATTEMPTS_PER_PUZZLE],
        dtick=max(1, ATTEMPTS_PER_PUZZLE // 8),
    ),
    yaxis=dict(title="Probability of at least one solution (%)", range=[0, 100]),
    font=dict(family="Arial, sans-serif", size=18),
    legend=dict(orientation="h", y=-0.22, x=0, groupclick="togglegroup"),
    margin=dict(l=85, r=25, t=70, b=140),
)
fig.show(config=PLOT_CONFIG)
fig.write_html(ARTIFACTS / "queens8_dfm_retries.html", config=PLOT_CONFIG)

np.savez_compressed(
    ARTIFACTS / "queens8_dfm_evaluation.npz",
    dataset_indices=evaluation_indices,
    puzzle_ids=puzzles["ids"][evaluation_indices],
    steps=np.asarray(EVAL_STEP_COUNTS),
    successes=np.stack([evaluation_successes[steps] for steps in EVAL_STEP_COUNTS]),
    attempt_counts=attempt_counts,
    pass_at_k=np.stack(
        [retry_results[steps]["pass_at_k"] for steps in EVAL_STEP_COUNTS]
    ),
    lower=np.stack([retry_results[steps]["lower"] for steps in EVAL_STEP_COUNTS]),
    upper=np.stack([retry_results[steps]["upper"] for steps in EVAL_STEP_COUNTS]),
    evaluation_seed=EVAL_SEED,
    batch_size=EVAL_BATCH_SIZE,
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    checkpoint_sha256=checkpoint_sha256,
    optimizer_updates=run.global_step,
    use_time_conditioning=model.use_time_conditioning,
)